# 01 — I tre paradigmi di domanda
- **Noul**: probabilità che un'affermazione sia vera.
- **Choice**: probabilità di ciascuna opzione dichiarata (una sola corretta).
- **Score**: probabilità di ciascun livello di una scala ordinata.

Le domande possono essere raggruppate in un'unica richiesta `/v1/systemone`.


In [ ]:
from decision_qa import to_systemone_payload, NoulQuestion, ChoiceQuestion, ScoreQuestion

qs = [
    NoulQuestion(label='is_true', statement='Paris is the capital of France.'),
    ChoiceQuestion(label='color', options=['red', 'green', 'blue']),
    ScoreQuestion(label='quality', rubric='Rate the answer quality.',
                  levels=['poor', 'fair', 'good']),
]
import json
print(json.dumps(to_systemone_payload({'question': 'Example'}, qs), indent=2))


## Confidenza e astensione
Una decisione è utile solo se sappiamo quando fidarci. Regola semplice: agisci quando la probabilità massima supera una soglia, altrimenti passa a un modello più forte o a una persona.


In [ ]:
def act(result, threshold=0.8):
    label = max(result.probabilities, key=result.probabilities.get)
    p = result.probabilities[label]
    return (label, 'commit') if p >= threshold else (label, 'escalate')

from decision_qa import MockClient
client = MockClient(seed=1)
res = client.decision({'question': 'Example'}, qs)[0]
print(res.probabilities, '->', act(res))


## Invarianza all'ordine
Un buon decision model dovrebbe dare la stessa risposta cambiando l'ordine delle opzioni. Usa `option_order_invariance` con una domanda etichettata; un mock senza gold restituisce probabilità uniformi, quindi i pareggi rendono arbitraria la risposta 'top'.


In [ ]:
from decision_qa.evaluate import option_order_invariance
from decision_qa import ChoiceQuestion
labelled = ChoiceQuestion(label='color', options=['red', 'green', 'blue'], gold_index=1)
print('order invariant:', option_order_invariance(client, {'question': 'Example'}, labelled))


Letture gratuite: [awesome-jev-prompts](https://github.com/vicfei/awesome-jev-prompts) (51 schemi di domande, CC0) e le [doc sui concetti](https://docs.typesafe.ai/introduction).
